# 1. DATA UNDERSTANDING

In [0]:
import pyspark.sql.functions as F


##  1.1 Carga Inicial de Datos

In [0]:
dfAccesoInternetOficiales = spark.read.csv(
    "/Volumes/catalogproyectosparkies/default/volumeproyectosparkies/Instituciones-Educativas-Oficiales-de-Boyacá-con-Acceso-a-Internet.csv",
    header=True, inferSchema=True
)
dfAccesoInternetOficiales.createOrReplaceTempView("dfAccesoInternetOficiales")
display(dfAccesoInternetOficiales)

## 1.2 Descripción de Datos

In [0]:
#Revisar las columnas para ver que carachas andamos viendo
dfAccesoInternetOficiales.printSchema()

In [0]:
#Cambiar el nombre de las columnas con - , en minúscula y sin tildes

df01 = dfAccesoInternetOficiales.withColumnRenamed("CÓDIGO DEPARTAMENTO","codigo-departamento").withColumnRenamed("DEPARTAMENTO","departamento").withColumnRenamed("PROVINCIA","provincia").withColumnRenamed("CÓDIGO MUNICIPIO","codigo-municipio").withColumnRenamed("MUNICIPIO","municipio").withColumnRenamed("CODIGO DANE INSTITUCION EDUCATIVA","codigo-dane-institucion-educativa").withColumnRenamed("NOMBRE INSTITUCION EDUCATIVA","institucion-educativa").withColumnRenamed("CODIGO DANE SEDE","codigo-dane-escuela").withColumnRenamed("NOMBRE SEDE EDUCATIVA","escuela").withColumnRenamed("ZONA","zona").withColumnRenamed("PROYECTOS DE CONECTIVIDAD 2024","proyectos-de-conectividad-2024").withColumnRenamed("OPERADOR","operador").withColumnRenamed("ESTADO","estado").withColumnRenamed("MEDIO DE ENLACE","medio-de-enlace").withColumnRenamed("ANCHO DE BANDA DE SUBIDA (MB)","ancho-de-banda-de-subida").withColumnRenamed("ANCHO DE BANDA DESCARGA (MB)","ancho-de-banda-descarga").withColumnRenamed("FINALIZACIÓN DEL CONTRATO","finalizacion-del-contrato").withColumnRenamed("LATITUD","latitud").withColumnRenamed("LONGITUD","longitud")


In [0]:
df01.display()

# 2. Exploración de Datos

In [0]:

display(df01.describe())

El dataframe tiene a las instituciones educativas de Boyacá con su conectividad, ubicación, proyectos de conectividad y operadores con su respectivo estado del servicio. 

Se cuenta con 2006 registros, donde existen 899 nulos en el medio de enlace,  1202 nulos en el ancho de subida, 902 nulos en el ancho de banda de descarga, 902 nulos en la finalización del contrato. Este número de 902 nulos en el ancho de banda de subida y de descarga junto al de finalización de contrato se debe a que son proyectos que no se han puedo en marcha (puede también ser relacionado al medio de enlace).

Solamente se le puede hacer un análisis a la banda de subida y a la de descarga. La primera tiene una media de 4.25 mbps de subida, siendo bastante límitado; mientras el de descarga está en 14.16. Ambos siendo muy bajos, demostrando falta de infraestructura y de calidad. Confirmado por la variación estándar que también es bastante limitada, lo que indicaría que la mayoría de las instituciones estarían bajo el mismo bajo estándar. 

El mínimo y máximo de ambas sigue demostrando lo mismo. Donde esta calidad de Internet no es justa para las instituciones educativas. 

In [0]:
df02 = df01.groupBy("zona").agg(F.avg("ancho-de-banda-descarga").alias("velocidad"))
display(df02)

Databricks visualization. Run in Databricks to view.

Se observa como es mejor la velocidad de descarga (para extraer datos de Internet) en colegios de zonas urbanas que en las rurales. 

In [0]:
df03 = df01.groupBy("zona").agg(F.avg("ancho-de-banda-de-subida").alias("velocidad_promedio_carga"))
display(df03)

Databricks visualization. Run in Databricks to view.

Estos resultados demuestran que no hay datos para la urbana de la velocidad de subida en zonas urbanas lo que es confirmado en el bloque de código de abajo. 

In [0]:
df01.filter(F.col("zona") == "URBANA") \
  .select("ancho-de-banda-de-subida") \
  .show()

In [0]:
df04 = df01.groupBy("medio-de-enlace").agg(F.avg("ancho-de-banda-descarga").alias("velocidad")).orderBy("velocidad", ascending=False)
display(df04)

Databricks visualization. Run in Databricks to view.

Se observa como hay un gran porcentaje de medio de enlace nulos que pueden ser proyectos sin información, que sin embargo tienen mayor velocidad que el resto de registros de medio de enlace y abarcan más del dataframe. Mientras que los proyectos satelitales, de fibra y de radio enlace tienen el promedio de 10 mbps;  que sigue siendo bastante bajo. Por lo que se deben depurar estos tipos de datos. 

In [0]:
df05 = df01.groupBy("operador").agg(F.avg("ancho-de-banda-descarga").alias("velocidad")).orderBy("velocidad", ascending=False)
display(df05)

En general el ancho de banda de descarga para el proyecto de "Alianza Pública para el Desarrollo Integral" es la más alta, seguida de "Unión Temporal de ETB", y finalmente la peor es la de "Colombia Más TV", mientras que los que no tienen operador tienen que ver con los que no tienen proyecto o no han iniciado.

In [0]:
df07 = df01.groupBy("estado").count()
display(df07)

Databricks visualization. Run in Databricks to view.

Se observa como hay más lugares que están pendientes de iniciar o sin servicio que los que realmente tienen operación, y ya se vio que los que tienen servicio tienen mala calidad de Internet. Por lo que se veo gran abandono total en el tema de acceso a Internet en colegios de Boyacá.

In [0]:
df08 = df01.groupBy("proyectos-de-conectividad-2024", "estado").count()
display(df08)

Se observa como existen muchísimos proyectos de las iniciativas de "Centro Digital" y de "Contectividad Escolar 2024" pendientes de inicio de la operación. Mientras que la mayoría se encuentran sin servicio.

In [0]:
df09 = df01.groupBy("municipio").agg(F.avg("ancho-de-banda-descarga").alias("velocidad_promedio")).orderBy("velocidad_promedio", ascending=False)

display(df09)


Databricks visualization. Run in Databricks to view.

Finalmente se observa la velocidad de descarga por departamento. Donde la mayoría se encuentran sobra el promedio entre 13 y 15 mbps en descarga. Se observa como los municipios de Corrales, Chiquinquirá y Monguí obtienen mejores resultados. Mientras que los peores son los de Chitaraque, Almeida, Guacamayas y la Victoria.

In [0]:
municipios = ["CORRALES", "CHIQUINQUIRÁ", "CHITARAQUE", "ALMEIDA", "GUACAMAYAS", "LA VICTORIA", "MONGUÍ"]

df010 = df01.filter(F.col("municipio").isin(municipios)).groupBy("municipio", "provincia").agg(F.avg("ancho-de-banda-descarga").alias("velocidad_promedio")).orderBy("velocidad_promedio", ascending=False)
display(df010)

Se evidencia que municipios como Almeida y Guacamayas que están ubicados en provincias más rurales como Oriente y Gutiérrez, presentan menores velocidades de conexión. Sin embargo, también se identifican casos como La Victoria y Chitaraque en provincias con mejor acceso geográfico o económicio, lo que puede significar que la calidad del servicio no depende únicamente de la ubicación geográfica, sino también de factores como el operador o el tipo de tecnología utilizada.

In [0]:
df01.groupBy(F.col("proyectos-de-conectividad-2024"),F.col("operador")).count().orderBy("count", ascending=False).show(truncate=False)
df01.groupBy(F.col("proyectos-de-conectividad-2024"),F.col("estado")).count().orderBy("count", ascending=False).show(truncate=False)
df01.groupBy("estado").count().orderBy("count", ascending=False).show(truncate=False)

De los dos tres bloques anteriores de código se observa que los que están sin proyecto no tienen servicio ni operador. Y se observa como hay más proyectos pendientes de inicio de operación y directamente sin servicio que los operativos. Demostrando una gran brecha de acceso a internet en ambientes educativos para el año 2024.

# 3. Reporte de Calidad de Datos

Se decide eliminar la columna de departamento y código de departamento como solamente vamos a trabajar con Boyacá.

In [0]:
df01.select(F.col("departamento")).distinct().show(truncate=False)

df01.select(F.col("provincia")).distinct().show(truncate=False)

df01.select(F.col("municipio")).distinct().show(truncate=False)

In [0]:
import pandas as pd

total = df01.count()

dfNulos = df01.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)for c in df01.columns])

nulos = dfNulos.collect()[0].asDict()

reporte = pd.DataFrame(list(nulos.items()), columns=["columna", "nulos"])
reporte["porcentaje"] = (reporte["nulos"] / total) * 100

display(reporte.sort_values(by="porcentaje", ascending=False))

Se observa que la mayoría de nulos se encuentran en el ancho de banda de subida, en finalización de contratos, de descarga, de enlace, longitud y latitud. De resto la información se ve completa. El de finalización de contrato está con varios vacios ya que hay instituciones que no tienen proyecto o no lo han comenzado. Mientras que el de ancho de banda de subida y de bajada puede ser por estos motivos o porque simplemente no se registró la información. Referente al medio de enlace pudo no haber sido registrada la información o no hay proyecto activo. Y finalmente la de latitud y longitud es nulo porque pueden existir colegios en zonas remotas que no tienen información. 

In [0]:
df01.select(F.col("proyectos-de-conectividad-2024")).distinct().show(truncate=False)

df01.select(F.col("operador")).distinct().show(truncate=False)

df01.select(F.col("estado")).distinct().show(truncate=False)

df01.select(F.col("medio-de-enlace")).distinct().show(truncate=False)

df01.select(F.col("finalizacion-del-contrato")).distinct().show(truncate=False)

Se observa que para 2024 se tienen 4 proyectos vigentes y existen 3 operadores. En el estados tenemos 4 posibles valores, el medio de enlace puede llegar a ser un indicativo de calidad y de la zona (si es de díficil acceso) para la conectividad de la instición. Se debe revisar porqué existe NA y NULL a tiempo. Y por último la finalización de contrato que se asume que es NULL para los que siguen vigente o para instituciones que no tienen aún contrato. 


Adicionalmente se observa como en estado está "PENDIENTE INICIO OPERACIÓN" Y "PENDIENTE INICIO DE OPERACIÓN" que es lo mismo, por lo que se procede a limpiar en la siguiente sección. 



In [0]:
df01.groupBy(F.col("medio-de-enlace"),F.col("proyectos-de-conectividad-2024")).count().orderBy("count", ascending=False).show(truncate=False)

Se observa como los medios de enlace se encuentran NULL y NA en los que no tienen proyecto. Se propone unificar bajo "SIN MEDIO" para este caso.
Mientras que para los "NA" de los que si tienen proyecto, se propone en dejarlos como "SIN DATOS" por lo que estós son los que están pendientes de implementación.

In [0]:
df01.filter(F.col("ancho-de-banda-de-subida").isNull()).groupBy("estado").count().show(truncate=False)
df01.filter(F.col("ancho-de-banda-descarga").isNull()).groupBy("estado").count().show(truncate=False)

Se observa que los que están pendientes de inicio, sin servicio y un outlier de operación no tienen el ancho de banda de subida. Y en general los que no tienen ancho de descarga están como nulos.

Se propone a dejarlos así ya que simplemente indican que no se ha comenzado o no tiene proyecto implementado. Por lo que si aporta información a nuestro negocio. Igual que la fecha de finalización, si no ha terminado y si no tiene proyecto asignado simplemente se deja el nulo. 

Adicionalmente se propone promediar el nulo que está "EN OPERACIÓN".

Y por último, para análisis posteriores estaría interesante tener otra columna que indique simplemente si tiene o no servicio de internet, por lo quer se crea su respectiva columna. 

In [0]:
df01.filter(F.col("latitud").isNull() | F.col("longitud").isNull()).groupBy("escuela").count().show(truncate=False)


Existen datos nulo no revisados los cuales corresponden a los nulos en latitud y longitud, que indican la ubicación geográfica. 
Como se observaron estos valores nulos, se decidió buscar a mano para encontrar las coordenadas y no eliminar las filas, ya que estos lugares probablemente son remotos y de díficil accesibilidad y por ello no fueron reconocidos en el dataframe.

- ESC RURAL LAS TAPIAS 5.589991979428556, -74.01218816598238
- CONCENTRACION JJPHN F KENNEDY 5.977274409017367, -74.59400727338647
- I.E SAN JOSE D ELA FLORIDA 5.287188696487662, -73.17368785709577
- MUSEÑITO 4.972267372899519, -73.31964859999255
- ESC LA LLANO 5.610780713223294, -73.28719870766237
- ESC DOS QUEBRADAS 4.40495806856695, -74.87481647116408
- SANTUARIO
- URB DE NIÑAS
- EL CARRIZAL 4.640452278501006, -73.65674351349223
- SANTA TERESA 5.626202840644254, -73.71309002569836
- PASCATA 5.324466684196802, -73.49058327042253
- VISTA HERMOSA 5.586662088011485, -72.92138212171537
- IE TECNICO AGROPECUARIO 5.713811536865067, -73.60212164771985
- ESC NORMAL SUPERIOR SAGRADO CORAZÓN 6.187545263635436, -72.47254657555203
- ESCUELA ANEXA
- ESC SANTA ROSA 5.7074376026561255, -74.00943497782059
- ESC LA CARRERA 
- SAN PEDRO CLAVER 6.023012129153874, -73.44667224747575

## 4. Filtros y Limpieza

### 4.1 Quitar departamento y código de departamento

In [0]:
if df01 is not None: df01 = df01.drop("departamento", "codigo-departamento")

### 4.2 Limpiar PENDIENTE INICIO OPERACIÓN Y PENDIENTE INICIO DE OPERACIÓN

In [0]:
df01 = df01.withColumn("estado",F.when((F.col("estado") == "PENDIENTE INICIO OPERACIÓN") |(F.col("estado") == "PENDIENTE INICIO DE OPERACIÓN"),"PENDIENTE INICIO OPERACION").otherwise(F.col("estado")))

df01.select(F.col("estado")).distinct().show(truncate=False)

#Ya quedó limpio :D

### 4.3 Correción MEDIO DE ENLACE

In [0]:
df01 = df01.withColumn("medio-de-enlace",
    F.when((F.col("proyectos-de-conectividad-2024") == "SIN PROYECTO") &(F.col("medio-de-enlace").isNull() | (F.col("medio-de-enlace") =="NA")),"SIN MEDIO"
    ).when((F.col("proyectos-de-conectividad-2024") != "SIN PROYECTO") &(F.col("medio-de-enlace") == "NA"),"SIN DATOS").otherwise(F.col("medio-de-enlace"))
)

In [0]:
df01.display()

### 4.4 Promedio DE ANCHO DE BANDA DE SUBIDA

In [0]:
promedioSubida = df01.filter((F.col("estado") == "EN OPERACIÓN") &(F.col("ancho-de-banda-de-subida").isNotNull())).agg(
F.avg("ancho-de-banda-de-subida")).collect()[0][0]

print(promedioSubida)

In [0]:
df01 = df01.withColumn("ancho-de-banda-de-subida",
    F.when((F.col("estado") == "EN OPERACIÓN") &(F.col("ancho-de-banda-de-subida").isNull()),promedioSubida).otherwise(F.col("ancho-de-banda-de-subida"))
)

In [0]:
#   Verificación de que si se cambió el outlier

df01.filter(F.col("ancho-de-banda-de-subida").isNull()).groupBy("estado").count().show(truncate=False)

### 4.5 Creación de Columna de Ancho de Banda

In [0]:
df01 = df01.withColumn("tiene_ancho_banda", F.when(F.col("ancho-de-banda-de-subida").isNull(), 0).otherwise(1))

df01.select(F.col("tiene_ancho_banda")).show(truncate=False)

#Creación de la nueva columna

In [0]:
df01.filter(F.col("finalizacion-del-contrato").isNull()).groupBy("estado").count().show(truncate=False)


In [0]:
df01.display()

In [0]:
df01 = df01.withColumn(
    "latitud",
    F.expr("try_cast(latitud as double)")
).withColumn(
    "longitud",
    F.expr("try_cast(longitud as double)")
)

### 4.6 Agregar Latitud y Longitud Manuales

In [0]:
df01 = df01.withColumn(
    "latitud",
    F.when(F.col("escuela") == "ESC RURAL LAS TAPIAS", "5.589991979428556")
     .when(F.col("escuela") == "CONCENTRACION JJPHN F KENNEDY", "5.977274409017367")
     .when(F.col("escuela") == "I.E SAN JOSE D ELA FLORIDA", "5.287188696487662")
     .when(F.col("escuela") == "MUSEÑITO", "4.972267372899519")
     .when(F.col("escuela") == "ESC LA LLANO", "5.610780713223294")
     .when(F.col("escuela") == "ESC DOS QUEBRADAS", "4.40495806856695")
     .when(F.col("escuela") == "EL CARRIZAL", "4.640452278501006")
     .when(F.col("escuela") == "SANTA TERESA", "5.626202840644254")
     .when(F.col("escuela") == "PASCATA", "5.324466684196802")
     .when(F.col("escuela") == "VISTA HERMOSA", "5.586662088011485")
     .when(F.col("escuela") == "IE TECNICO AGROPECUARIO", "5.713811536865067")
     .when(F.col("escuela") == "ESC NORMAL SUPERIOR SAGRADO CORAZÓN", "6.187545263635436")
     .when(F.col("escuela") == "ESC SANTA ROSA", "5.7074376026561255")
     .when(F.col("escuela") == "SAN PEDRO CLAVER", "6.023012129153874")
     .otherwise(F.col("latitud"))
)

df01 = df01.withColumn(
    "longitud",
    F.when(F.col("escuela") == "ESC RURAL LAS TAPIAS", "-74.01218816598238")
     .when(F.col("escuela") == "CONCENTRACION JJPHN F KENNEDY", "-74.59400727338647")
     .when(F.col("escuela") == "I.E SAN JOSE D ELA FLORIDA", "-73.17368785709577")
     .when(F.col("escuela") == "MUSEÑITO", "-73.31964859999255")
     .when(F.col("escuela") == "ESC LA LLANO", "-73.28719870766237")
     .when(F.col("escuela") == "ESC DOS QUEBRADAS", "-74.87481647116408")
     .when(F.col("escuela") == "EL CARRIZAL", "-73.65674351349223")
     .when(F.col("escuela") == "SANTA TERESA", "-73.71309002569836")
     .when(F.col("escuela") == "PASCATA", "-73.49058327042253")
     .when(F.col("escuela") == "VISTA HERMOSA", "-72.92138212171537")
     .when(F.col("escuela") == "IE TECNICO AGROPECUARIO", "-73.60212164771985")
     .when(F.col("escuela") == "ESC NORMAL SUPERIOR SAGRADO CORAZÓN", "-72.47254657555203")
     .when(F.col("escuela") == "ESC SANTA ROSA", "-74.00943497782059")
     .when(F.col("escuela") == "SAN PEDRO CLAVER", "-73.44667224747575")
     .otherwise(F.col("longitud"))
)

In [0]:
df01 = df01.dropna(subset=["latitud", "longitud"])

## 4.7 Resultado Final

In [0]:
total = df01.count()

dfNulos = df01.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)for c in df01.columns])

nulos = dfNulos.collect()[0].asDict()

reporte = pd.DataFrame(list(nulos.items()), columns=["columna", "nulos"])
reporte["porcentaje"] = (reporte["nulos"] / total) * 100

display(reporte.sort_values(by="porcentaje", ascending=False))

In [0]:
df01.display()

In [0]:
display(df01)